# Labor forecast workflow conversion

Pandas notebook translating the Alteryx workflow a section at a time. Current and prior forecasts are prepared and joined below; later branches and outputs remain to be captured.

## Current Labor Forecast input

Read `Data - Forecast & Opps` from the workbook in `.env` key `currentlaborpath`; keep its filename.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
current_labor_path = Path(os.environ["currentlaborpath"])
current_labor_raw = pd.read_excel(
    current_labor_path,
    sheet_name="Data - Forecast & Opps",
    header=0,
)
current_labor_raw["FileName"] = current_labor_path.name
print(len(current_labor_raw), "current labor forecast rows loaded")

## Current GLC and months

Trim `GLC`; transpose month columns into rows, excluding `Labor Forecast Total`.

In [ ]:
current_labor_raw["GLC"] = current_labor_raw["GLC"].astype("string").str.strip()

key_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
]
missing_keys = [column for column in key_columns if column not in current_labor_raw.columns]
if missing_keys:
    raise KeyError(f"Missing current forecast key columns: {missing_keys}")

value_columns = [
    column for column in current_labor_raw.columns
    if column not in key_columns and column != "Labor Forecast Total"
]
current_labor = current_labor_raw.melt(
    id_vars=key_columns,
    value_vars=value_columns,
    var_name="Name",
    value_name="Value",
)
print(len(current_labor), "transposed rows from", len(value_columns), "data columns")

## Current month flag

`DoesHaveMonth` checks whether `Name` contains a digit so the forecast total is not treated as a month. Trim surrounding whitespace from `Segment`.

In [ ]:
current_labor["DoesHaveMonth"] = (
    current_labor["Name"].astype("string").str.contains(r"\d+", regex=True, na=False)
)
current_labor["Segment"] = current_labor["Segment"].astype("string").str.strip().str.slice(0, 255)
current_labor[["Name", "Value", "DoesHaveMonth", "Segment"]].head()

## Current month and category

Remove `Hours` from the month/year for date conversion; split `Name` on spaces and use its third part as `Category` for monthly hours. Rename `Value` to `NEW Hours`, remove temporary fields, and keep non-null `Type`.

In [ ]:
# RegEx (96) and DateTime (97)
current_labor["RegExOut1"] = current_labor["Name"].astype("string").str.extract(
    r"(\w+ \w+)", expand=False
)
current_labor["DateTime_Out"] = pd.to_datetime(
    current_labor["RegExOut1"].str.replace(
        r"^([A-Za-z]{3})[A-Za-z]+(?= \\d{2}$)", r"\\1", regex=True
    ),
    format="%b %y", errors="coerce"
)

# Select (103) and Text To Columns (318)
current_labor = current_labor.rename(columns={"Value": "NEW Hours"})
current_labor["NEW Hours"] = pd.to_numeric(current_labor["NEW Hours"], errors="coerce")
name_parts = current_labor["Name"].astype("string").str.split(" ", n=2, expand=True)
third_word = name_parts[2] if 2 in name_parts.columns else pd.Series(
    pd.NA, index=current_labor.index, dtype="string"
)

# Formula (100): Alteryx uses -1 for a true RegEx_Match.
current_labor["Category"] = current_labor["Name"].astype("string").where(
    ~current_labor["DoesHaveMonth"], third_word
)

# Select (101) and true output of Filter (102)
current_labor_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
    "NEW Hours", "RegExOut1", "DateTime_Out", "Category",
]
current_labor = current_labor.loc[
    current_labor["Type"].notna(), current_labor_columns
].copy()
current_labor.head()

## Prior Labor Forecast input and months

Read `Labor Forecast - Prior.xlsx` beside the current workbook, trim `GLC`, and transpose September 2026–August 2027 hours. Exclude `Labor Forecast Total`; warn about missing months.

In [ ]:
import warnings

prior_labor_path = current_labor_path.with_name("Labor Forecast - Prior.xlsx")
prior_labor_raw = pd.read_excel(
    prior_labor_path,
    sheet_name="Data - Forecast & Opps",
    header=0,
)
prior_labor_raw["FileName"] = prior_labor_path.name
prior_labor_raw["GLC"] = prior_labor_raw["GLC"].astype("string").str.strip().str.slice(0, 255)

prior_key_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
]
missing_prior_keys = [
    column for column in prior_key_columns if column not in prior_labor_raw.columns
]
if missing_prior_keys:
    raise KeyError(f"Missing prior forecast key columns: {missing_prior_keys}")

prior_month_columns = [
    "Sep 26 Hours", "Oct 26 Hours", "Nov 26 Hours", "Dec 26 Hours",
    "Jan 27 Hours", "Feb 27 Hours", "Mar 27 Hours", "Apr 27 Hours",
    "May 27 Hours", "June 27 Hours", "July 27 Hours", "Aug 27 Hours",
]
missing_prior_months = [
    column for column in prior_month_columns if column not in prior_labor_raw.columns
]
if missing_prior_months:
    warnings.warn(f"Prior forecast month columns missing: {missing_prior_months}")
present_prior_months = [
    column for column in prior_month_columns if column in prior_labor_raw.columns
]
prior_labor = prior_labor_raw.melt(
    id_vars=prior_key_columns,
    value_vars=present_prior_months,
    var_name="Name",
    value_name="Value",
)
print(len(prior_labor), "prior forecast rows from", len(present_prior_months), "month columns")


## Prior month and category

Repeat the month parsing and whitespace cleanup, rename `Value` to `Hours`, then keep non-null `Type`.

In [ ]:
# Formula (50), RegEx (51), and DateTime (52)
prior_labor["DoesHaveMonth"] = (
    prior_labor["Name"].astype("string").str.contains(r"\d+", regex=True, na=False)
)
prior_labor["Segment"] = prior_labor["Segment"].astype("string").str.strip().str.slice(0, 255)
prior_labor["RegExOut1"] = prior_labor["Name"].astype("string").str.extract(
    r"(\w+ \w+)", expand=False
)
prior_labor["DateTime_Out"] = pd.to_datetime(
    prior_labor["RegExOut1"].str.replace(
        r"^([A-Za-z]{3})[A-Za-z]+(?= \\d{2}$)", r"\\1", regex=True
    ),
    format="%b %y", errors="coerce"
)

# Select (58), Text To Columns (54), and Formula (55)
prior_labor = prior_labor.rename(columns={"Value": "Hours"})
prior_labor["Hours"] = pd.to_numeric(prior_labor["Hours"], errors="coerce")
prior_name_parts = prior_labor["Name"].astype("string").str.split(" ", n=2, expand=True)
prior_third_word = (
    prior_name_parts[2] if 2 in prior_name_parts.columns
    else pd.Series(pd.NA, index=prior_labor.index, dtype="string")
)
prior_labor["Category"] = prior_labor["Name"].astype("string").where(
    ~prior_labor["DoesHaveMonth"], prior_third_word
)

# Select (56) and the true output of Filter (57)
prior_labor_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
    "Hours", "RegExOut1", "DateTime_Out", "Category",
]
prior_labor = prior_labor.loc[
    prior_labor["Type"].notna(), prior_labor_columns
].copy()
prior_labor.head()


## Joining Prior and Current

Join on `Program`, `GLC`, month/year (`RegExOut1`), `Type`, and `Firm/FO-SS/New`. Keep matched rows plus unmatched rows whose `Category` is `Hours`; the latter removes total values without a match. Union by field name. The prior forecast retains its `Hours` name, and matched prior fields carry `Right_` prefixes.

In [ ]:
join_fields = ["Program", "GLC", "RegExOut1", "Type", "Firm/FO-SS/New"]
current_has_keys = current_labor[join_fields].notna().all(axis=1)
prior_has_keys = prior_labor[join_fields].notna().all(axis=1)

# Alteryx does not join null keys; keep those rows for the unmatched outputs.
comparison = current_labor.loc[current_has_keys].merge(
    prior_labor.loc[prior_has_keys].add_prefix("Right_"),
    how="outer",
    left_on=join_fields,
    right_on=[f"Right_{field}" for field in join_fields],
    indicator=True,
    sort=False,
)
left_fields = current_labor.columns.tolist()
right_fields = [f"Right_{field}" for field in prior_labor.columns]

current_unmatched = pd.concat(
    [
        comparison.loc[comparison["_merge"].eq("left_only"), left_fields],
        current_labor.loc[~current_has_keys],
    ],
    ignore_index=True,
)
prior_unmatched = pd.concat(
    [
        comparison.loc[comparison["_merge"].eq("right_only"), right_fields].rename(
            columns=lambda column: column.removeprefix("Right_")
        ),
        prior_labor.loc[~prior_has_keys],
    ],
    ignore_index=True,
)
current_unmatched_hours = current_unmatched.loc[
    current_unmatched["Category"].eq("Hours").fillna(False)
]
prior_unmatched_hours = prior_unmatched.loc[
    prior_unmatched["Category"].eq("Hours").fillna(False)
]
matched_forecasts = comparison.loc[comparison["_merge"].eq("both")].drop(
    columns="_merge"
)

combined_forecasts = pd.concat(
    [current_unmatched_hours, matched_forecasts, prior_unmatched_hours],
    ignore_index=True,
    sort=False,
)
print(
    len(matched_forecasts), "matched,",
    len(current_unmatched_hours), "current only,",
    len(prior_unmatched_hours), "prior only"
)
combined_forecasts.head()
